# Interactive 3D simplex-pair explorer

This notebook samples a cached protein and a rooted pair of cliques from its contact graph. The shared root and feature residues are highlighted in an interactive 3D C-alpha view; the rest of the protein graph stays translucent.

- **Random protein** loads another graph and its cached mmCIF coordinates.
- **Random feature** samples a new pair for the current protein and slider settings.
- **Representative distance** selects the exact shortest-path distance, in graph hops, between the two simplex representatives, matching notebook 03's vectorizer.
- **Maximum simplex size** sets the largest simplex in the sampled pair. Each simplex may be smaller, and the pair may mix sizes.

Drag the plot to orbit, right-drag to pan, and scroll to zoom. The graph and coordinates come from notebooks 01 and 02 caches.

In [ ]:
from pathlib import Path
import random
import sys

import ipywidgets as widgets
from IPython.display import Markdown, clear_output, display

PROJECT_ROOT = next(
    (path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (path / "notebooks" / "src" / "enzyme_workflow").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Could not find the repository root from this notebook's working directory")
sys.path.insert(0, str(PROJECT_ROOT / "notebooks" / "src"))

from enzyme_workflow.common import add_project_sources
from enzyme_workflow.feature_explorer import (
    draw_protein_feature,
    load_available_proteins,
    load_protein_geometry,
    sample_rooted_simplex_pair,
)

PROJECT_ROOT = add_project_sources(PROJECT_ROOT)
PROTEINS = load_available_proteins(PROJECT_ROOT)
print(f"Available cached proteins: {len(PROTEINS):,}")

## Controls and 3D graph

Changing either slider samples a feature that matches the new setting in the current protein. If the selected protein has no matching pair, the 3D graph remains visible and the message explains which setting to change.

In [ ]:
rng = random.Random()
state = {"row": None, "graph": None, "coordinates": None, "feature": None}
plot_output = widgets.Output(layout=widgets.Layout(width="100%"))

protein_button = widgets.Button(description="Random protein", button_style="primary")
feature_button = widgets.Button(description="Random feature")
distance_slider = widgets.IntSlider(
    value=1,
    min=0,
    max=2,
    step=1,
    description="Representative distance (hops)",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="48%"),
)
size_slider = widgets.IntSlider(
    value=4,
    min=1,
    max=4,
    step=1,
    description="Maximum simplex size (vertices)",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="48%"),
)


def _feature_title(row, feature):
    left_label = ", ".join(str(state["graph"].nodes[node].get("label", "?")) for node in feature.left)
    right_label = ", ".join(str(state["graph"].nodes[node].get("label", "?")) for node in feature.right)
    return (
        f"<b>{row['sample_id']}</b> · EC {row.get('ec_number', '?')} · "
        f"{row.get('protein_name', 'Protein')}<br>"
        f"Feature: {len(feature.left)}-vertex simplex [{left_label}] ↔ "
        f"{len(feature.right)}-vertex simplex [{right_label}] · "
        f"shared root {feature.root} · representative distance {feature.distance} hops"
    )


def _render(message=None):
    with plot_output:
        clear_output(wait=True)
        if message:
            display(Markdown(message))
        if state["graph"] is None:
            return
        title = (
            _feature_title(state["row"], state["feature"])
            if state["feature"] is not None
            else f"<b>{state['row']['sample_id']}</b> · No matching feature selected"
        )
        figure = draw_protein_feature(
            state["graph"], state["coordinates"], state["feature"], title=title
        )
        display(figure)


def _sample_feature(_=None):
    if state["graph"] is None:
        return
    state["feature"] = sample_rooted_simplex_pair(
        state["graph"],
        max_simplex_size=size_slider.value,
        distance=distance_slider.value,
        rng=rng,
    )
    if state["feature"] is None:
        _render(
            f"No pair at distance **{distance_slider.value}** with simplex sizes up to "
            f"**{size_slider.value}** for {state['row']['sample_id']}. Change a slider or pick another protein."
        )
    else:
        _render()


def _random_protein(_=None):
    state.update(row=None, graph=None, coordinates=None, feature=None)
    indices = list(range(len(PROTEINS)))
    rng.shuffle(indices)
    failures = []
    for index in indices[: min(25, len(indices))]:
        row = PROTEINS.iloc[index]
        try:
            graph, coordinates = load_protein_geometry(row)
        except Exception as exc:
            failures.append(f"{row['sample_id']}: {exc}")
            continue
        state.update(row=row, graph=graph, coordinates=coordinates)
        _sample_feature()
        return
    details = "<br>".join(failures[:5])
    _render(
        "Could not load a cached protein geometry from the sampled entries. "
        "Run notebook 02 again for the active manifest. " + details
    )


protein_button.on_click(_random_protein)
feature_button.on_click(_sample_feature)
distance_slider.observe(_sample_feature, names="value")
size_slider.observe(_sample_feature, names="value")

display(
    widgets.VBox(
        [
            widgets.HBox([protein_button, feature_button]),
            widgets.HBox([distance_slider, size_slider]),
            plot_output,
        ],
        layout=widgets.Layout(width="100%"),
    )
)
_random_protein()